In [0]:
%sql
-- Set catalog
USE CATALOG labour_market_platform;
-- Create schema if not exists
CREATE SCHEMA IF NOT EXISTS silver_jobtech;
-- Use schema
USE SCHEMA silver_jobtech;

In [0]:
%sql

SELECT *
FROM bronze_jobtech.current_ads_raw
LIMIT 10;

In [0]:
%sql

DESCRIBE bronze_jobtech.current_ads_raw;

From the inspections above we can already confirm that:
- current_ads_raw is already flattened at the top level
- but it still contains several **nested structs / arrays** such as:
    - employer
    - occupation
    - occupation_field
    - occupation_group
    - salary_type
    - scope_of_work
    - working_hours_type
    - workplace_address
    - must_have
    - nice_to_have
That means Silver should be the layer where we:
- extract the business-useful scalar fields
- standardize names
- align current + historical into one canonical job-posting shape

But before we do that, we need to confirm whether  **historical** has the same structure.

In [0]:
%sql
DESCRIBE bronze_jobtech.historical_ads_raw

## Common core between current and historical

Both tables share the main job-posting fields we need for Silver, such as:

- id
- headline
- description
- application_deadline
- duration
- employer
- employment_type
- occupation
- occupation_field
- occupation_group
- number_of_vacancies
- publication_date
- last_publication_date
- salary_type
- scope_of_work
- working_hours_type
- workplace_address
- webpage_url
- removed
- removed_date

## Schema differences

Historical has some extra fields like:

- detected_language
- keywords
- remote_work
- open_for_all
- trainee
- franchise
- hire_work_place

Current has one field not in historical:

- relevance

That means the right Silver strategy is:

- build one canonical Silver table
- keep the shared business fields
- flatten the important nested structs
- add a record_source column so we know whether the row came from historical or current

In [0]:
%sql
SELECT
    id AS job_id,
    headline AS job_title,
    description,
    employer.name AS employer_name,
    occupation[0].label AS occupation_label,
    occupation_field[0].label AS occupation_field_label,
    occupation_group[0].label AS occupation_group_label,
    workplace_address.municipality_code AS municipality_code,
    workplace_address.city AS city,
    employment_type[0].label AS employment_type_label,
    duration.label AS duration_label,
    scope_of_work.min AS scope_of_work_min,
    scope_of_work.max AS scope_of_work_max,
    working_hours_type.label AS working_hours_type_label,
    salary_type.label AS salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    'historical' AS record_source
FROM bronze_jobtech.historical_ads_raw
LIMIT 10;

In [0]:
%sql
-- Preview unified Silver JobTech dataset
SELECT
    id AS job_id,
    headline AS job_title,
    description,
    employer.name AS employer_name,
    element_at(occupation, 1).label AS occupation_label,
    element_at(occupation_field, 1).label AS occupation_field_label,
    element_at(occupation_group, 1).label AS occupation_group_label,
    workplace_address.municipality_code AS municipality_code,
    workplace_address.city AS city,
    element_at(employment_type, 1).label AS employment_type_label,
    duration.label AS duration_label,
    scope_of_work.min AS scope_of_work_min,
    scope_of_work.max AS scope_of_work_max,
    working_hours_type.label AS working_hours_type_label,
    salary_type.label AS salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    'historical' AS record_source
FROM bronze_jobtech.historical_ads_raw

UNION ALL

SELECT
    id AS job_id,
    headline AS job_title,
    description,
    employer.name AS employer_name,
    occupation.label AS occupation_label,
    occupation_field.label AS occupation_field_label,
    occupation_group.label AS occupation_group_label,
    workplace_address.municipality_code AS municipality_code,
    workplace_address.city AS city,
    employment_type.label AS employment_type_label,
    duration.label AS duration_label,
    scope_of_work.min AS scope_of_work_min,
    scope_of_work.max AS scope_of_work_max,
    working_hours_type.label AS working_hours_type_label,
    salary_type.label AS salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    'current' AS record_source
FROM bronze_jobtech.current_ads_raw
LIMIT 20;